<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/05b_promptfoo_owasp_agentic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5b: Red-Teaming, OWASP Top 10 for Agentic Applications (2026) Plus Crescendo and Multilingual Strategies

**Goal:** Extend the Phase 5a red-team layer with the OWASP Top 10 for Agentic Applications (2026) preset, and add crescendo multi-turn and multilingual attack strategies, both absent from Phase 5a's single-turn, English-only payload set. Compare the combined detection picture against Project 1's 28% baseline and Phase 5a's own results.

**Tools:** Promptfoo 0.121.19 (Node), OWASP Top 10 for Agentic Applications (2026)

**OWASP Top 10 for Agentic Applications (2026) categories tested:**
- AAI01: Agent Authorization and Control Hijacking
- AAI02: Agent Critical Systems Interaction
- AAI03: Agent Goal and Instruction Manipulation
- AAI04: Agent Hallucination Exploitation
- AAI05: Agent Impact Chain and Blast Radius
- AAI06: Agent Memory and Context Manipulation
- AAI07: Agent Orchestration and Multi-Agent Exploitation
- AAI08: Agent Resource and Service Exhaustion
- AAI09: Agent Supply Chain and Dependency Attacks
- AAI10: Agent Untraceability

**New attack strategies (absent from Phase 5a):**
- **Crescendo:** multi-turn escalation, each turn individually looks benign, the cumulative sequence achieves what a single-turn payload could not.
- **Multilingual:** the same attack intent expressed in a non-English language, testing whether detection depends on English-language pattern matching.

**Project 1 / Phase 5a connection:** Phase 5a tested single-turn, English-only payloads against the non-agentic baseline pipeline and found four categories structurally undetectable as failures (no attack surface existed for them). This phase asks a harder question: since the baseline pipeline still has no real agentic capability (no tool use, no multi-agent orchestration, no persistent memory across sessions), most OWASP Agentic categories are expected to be structurally absent here too, and that expectation itself is a finding worth stating plainly rather than treating as success.

**SIMULATED_OUTPUT flag:** Set to True. Promptfoo configuration is real and validated against the actual CLI (`promptfoo validate config`, confirmed working in Phase 5a). Full scan runs when API credits are available.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm Phase 5a

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase5a_path = DRIVE_PATH + "phase05a_promptfoo_owasp_results.json"
if os.path.exists(phase5a_path):
    with open(phase5a_path) as f:
        phase5a = json.load(f)
    print("Phase 5a results confirmed.")
    print(f"  Detection rate: {phase5a['detection_rate']:.0%} "
          f"({phase5a['detected_count']}/{phase5a['attack_case_count']})")
    print(f"  Structural (no attack surface): "
          f"{len(phase5a['structurally_absent_categories'])}")
else:
    print("WARNING: Phase 5a results not found.")
    print(f"Expected: {phase5a_path}")
    print("Run 05a_promptfoo_owasp_llm.ipynb first.")

Mounted at /content/drive
Phase 5a results confirmed.
  Detection rate: 100% (10/10)
  Structural (no attack surface): 4


In [2]:
# Cell 3: Install packages

# Check Node version first. Promptfoo 0.121.19 requires Node ^20.20.0 or
# >=22.22.0. Colab's default preinstalled Node (v20.19.0) is just under
# this, which is why Phase 5a needed a manual upgrade. If this is a fresh
# runtime, that upgrade may not have persisted, so this checks and
# upgrades again if needed rather than assuming it carried over.

import subprocess

node_version = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
print("Current Node version:", node_version)

major_minor = node_version.lstrip("v").split(".")
major, minor = int(major_minor[0]), int(major_minor[1])

needs_upgrade = not ((major == 20 and minor >= 20) or major >= 22)

if needs_upgrade:
    print("Node version insufficient for Promptfoo 0.121.19. Upgrading to Node 22.x...")
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    !node --version
else:
    print("Node version sufficient, no upgrade needed.")

!npm install -g promptfoo@0.121.19 --silent
!pip install langfuse --quiet

print("Packages installed.")
print("promptfoo 0.121.19 (Node-based, installed via npm)")

os.environ["PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION"] = "true"

Current Node version: v20.19.0
Node version insufficient for Promptfoo 0.121.19. Upgrading to Node 22.x...
v22.23.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
promptfoo 0.121.19 (Node-based, installed via npm